In [2]:
import sys, os

if "google.colab" in sys.modules and not os.path.exists(".setup_complete"):
    # Install xvfb and our launcher script for it
    !apt-get install -y xvfb
    !wget -q https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/xvfb -O ../xvfb

    # Download dependencies from Github
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/atari_wrappers.py
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/env_batch.py
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/runners.py

    # Update the gym environment to be compatible with the Atari environment
    !pip install -q gymnasium[atari,accept-rom-license]
    !pip install -q tensorboardX

    !touch .setup_complete

# This code creates a virtual display to draw game images on.
# It will have no effect if your machine has a monitor.
if type(os.environ.get("DISPLAY")) is not str or len(os.environ.get("DISPLAY")) == 0:
    !bash ../xvfb start
    os.environ["DISPLAY"] = ":1"

os.environ["SDL_AUDIODRIVER"] = "dummy"

Starting virtual X frame buffer: Xvfb.


# Implementing Advantage-Actor Critic (A2C)

In this notebook you will implement Advantage Actor Critic algorithm that trains on a batch of Atari 2600 environments running in parallel.

Firstly, we will use environment wrappers implemented in file `atari_wrappers.py`. These wrappers preprocess observations (resize, grayscale, take max between frames, skip frames and stack them together) and rewards. Some of the wrappers help to reset the environment and pass `done` flag equal to `True` when agent dies.
File `env_batch.py` includes implementation of `ParallelEnvBatch` class that allows to run multiple environments in parallel. To create an environment we can use `nature_dqn_env` function. Note that if you are using
PyTorch and not using `tensorboardX` you will need to implement a wrapper that will log **raw** total rewards that the *unwrapped* environment returns and redefine the implemention of `nature_dqn_env` function here.



In [ ]:
import numpy as np
import gymnasium as gym
from atari_wrappers import nature_dqn_env


env_name = "SpaceInvadersNoFrameskip-v4"
nenvs = 32  # change this if you have more than 8 CPU ;)
summaries = "Tensorboard"

env = nature_dqn_env(env_name, nenvs=nenvs, summaries=summaries)
obs, _ = env.reset()
assert obs.shape == (nenvs, 4, 84, 84)
assert obs.dtype == np.float32


A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]
A.L.E: Arcade Learning Environment (version 0.8.1+53

Process Process-40:
Process Process-34:
Process Process-33:
Process Process-2:
Process Process-38:
Process Process-1:
Process Process-26:
A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]
Process Process-35:
Process Process-39:
A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]
Process Process-13:
A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]
Process Process-5:
Process Process-16:
Process Process-30:
A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]
Process Process-20:
Process Process-24:
Process Process-32:
A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]
Process Process-14:
Process Process-21:
Process Process-7:
Process Process-29:
Process Process-4:
A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]
Process Process-8:
A.L.E: Arcade Learning Environment (version 0.8.1+53f58b7)
[Powered by Stella]
Process Proce

Next, we will need to implement a model that predicts logits and values. It is suggested that you use the same model as in [Nature DQN paper](https://www.nature.com/articles/nature14236) with a modification that instead of having a single output layer, it will have two output layers taking as input the output of the last hidden layer. **Note** that this model is different from the model you used in homework where you implemented DQN. You can use your favorite deep learning framework here. We suggest that you use orthogonal initialization with parameter $\sqrt{2}$ for kernels and initialize biases with zeros.

In [ ]:
N_FRAMES_STACKED = 4
N_ACTIONS = int(env.action_space.n)
N_ACTIONS

6

In [ ]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [5]:
from typing import Tuple


class ConvBackbone(nn.Sequential):
    def __init__(self, c_in: int = N_FRAMES_STACKED) -> None:
        super().__init__(
            nn.Conv2d(in_channels=c_in, out_channels=32, kernel_size=8, stride=4, padding=0),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=4, stride=2, padding=0),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=0),
            nn.ReLU(inplace=True),
            nn.Flatten(),
        )


class A2CHead(nn.Module):
    def __init__(self, n_actions=N_ACTIONS, inp_size=64 * 7 * 7, hidden_size=512) -> None:
        super().__init__()
        self.value_stream = nn.Sequential(
            nn.Linear(inp_size, hidden_size), nn.ReLU(inplace=True), nn.Linear(hidden_size, 1)
        )
        self.action_stream = nn.Sequential(
            nn.Linear(inp_size, hidden_size),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_size, n_actions),
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        values = self.value_stream(x).squeeze(-1)
        action_logits = self.action_stream(x)
        return values, action_logits


class A2CModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = ConvBackbone()
        self.heads = A2CHead()

    def forward(self, x):
        features = self.backbone(x)
        return self.heads(features)

You will also need to define and use a policy that wraps the model. While the model computes logits for all actions, the policy will sample actions and also compute their log probabilities.  `policy.act` should return a dictionary of all the arrays that are needed to interact with an environment and train the model.
 Note that actions must be an `np.ndarray` while the other
tensors need to have the type determined by your deep learning framework.

In [6]:
class Policy:
    def __init__(self, model: A2CModel, device):
        self.model = model
        self.device = device

    def act(self, inputs):
        # Implement a policy by calling the model, sampling actions and computing their log probs.
        # Should return a dict containing keys ['actions', 'logits', 'log_probs', 'values'].
        inputs = np.asarray(inputs)
        inputs = torch.from_numpy(inputs).to(self.device)
        values, action_logits = self.model(inputs)
        probs = torch.softmax(action_logits, dim=-1)
        actions = torch.multinomial(probs, num_samples=1).squeeze(-1)
        log_probs = torch.log_softmax(action_logits, dim=-1)
        log_probs_for_sampled_actions = log_probs[
            torch.arange(actions.shape[0], device=actions.device), actions
        ]
        return {
            "actions": actions.cpu().numpy(),
            "logits": action_logits,
            "log_probs": log_probs_for_sampled_actions,
            "values": values,
        }

In [7]:
import torch.nn as nn

def orthogonal_init(module, gain=2**0.5):
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        nn.init.orthogonal_(module.weight, gain=gain)
        if module.bias is not None:
            nn.init.constant_(module.bias, 0.0)


Next will pass the environment and policy to a runner that collects partial trajectories from the environment.
The class that does is is already implemented for you.

In [8]:
from runners import EnvRunner

This runner interacts with the environment for a given number of steps and returns a dictionary containing
keys

* 'observations'
* 'rewards'
* 'resets'
* 'actions'
* all other keys that you defined in `Policy`

under each of these keys there is a python `list` of interactions with the environment. This list has length $T$ that is size of partial trajectory. Partial trajectory for given moment `t` is part of `ComputeValueTargets.__call__` input argument `trajectory` from moment `t` to the end (i.e. it's different at each iteration in the algorithm).

To train the part of the model that predicts state values you will need to compute the value targets.
Any callable could be passed to `EnvRunner` to be applied to each partial trajectory after it is collected.
Thus, we can implement and use `ComputeValueTargets` callable.
The formula for the value targets is simple:

$$
\hat v(s_t) = \left( \sum_{t'=0}^{T - 1} \gamma^{t'}r_{t+t'} \right) + \gamma^T \hat{v}(s_{t+T}),
$$

In implementation, however, do not forget to use
`trajectory['resets']` flags to check if you need to add the value targets at the next step when
computing value targets for the current step. You can access `trajectory['state']['latest_observation']`
to get last observations in partial trajectory &mdash; $s_{t+T}$.

In [9]:
class ComputeValueTargets:
    def __init__(self, policy, gamma=0.99):
        self.policy = policy
        self.gamma = gamma

    @torch.no_grad()
    def __call__(self, trajectory):
        """Compute value targets for a given partial trajectory."""

        # This method should modify trajectory inplace by adding
        # an item with key 'value_targets' to it.
        last_obs = trajectory["state"]["latest_observation"]
        last_obs_values = self.policy.act(last_obs)["values"]
        n_steps = trajectory["state"]["env_steps"]
        n_envs = last_obs.shape[0]
        value_targets = []
        rewards = trajectory["rewards"]
        resets = trajectory["resets"]
        cur_partial_returns = np.zeros(n_envs)
        cur_second_terms = last_obs_values.cpu().numpy()
        for idx in range(n_steps - 1, -1, -1):
            reset = resets[idx]
            cur_partial_returns = rewards[idx] + self.gamma * (1 - reset) * cur_partial_returns
            cur_second_terms = self.gamma * (1 - reset) * cur_second_terms
            value_targets.append(cur_partial_returns + cur_second_terms)
        trajectory["value_targets"] = list(reversed(value_targets))

After computing value targets we will transform lists of interactions into tensors
with the first dimension `batch_size` which is equal to `env_steps * num_envs`, i.e. you essentially need
to flatten the first two dimensions.

In [10]:
class MergeTimeBatch:
    """Merges first two axes typically representing time and env batch."""

    def __init__(self, device):
        self.device = device

    def __call__(self, trajectory):
        # Modify trajectory inplace
        for key, value in trajectory.items():
            transformed_value = None
            if key in ["logits", "values", "log_probs"]:
                assert torch.is_tensor(value[0])
                transformed_value = torch.stack(value)
            elif key == "value_targets":
                assert isinstance(value[0], np.ndarray)
                transformed_value = torch.as_tensor(np.asarray(value), device=self.device)
            if transformed_value is not None:
                trajectory[key] = transformed_value.flatten(0, 1)

In [11]:
model = A2CModel().to(device)
model.apply(orthogonal_init)
policy = Policy(model, device)
runner = EnvRunner(
    env=env,
    policy=policy,
    nsteps=5,
    transforms=[
        ComputeValueTargets(policy),
        MergeTimeBatch(device),
    ],
)


Now is the time to implement the advantage actor critic algorithm itself. You can look into your lecture,
[Mnih et al. 2016](https://arxiv.org/abs/1602.01783) paper, and [lecture](https://www.youtube.com/watch?v=Tol_jw5hWnI&list=PLkFD6_40KJIxJMR-j5A1mkxK26gh_qg37&index=20) by Sergey Levine.

In [12]:
from typing import Dict, Tuple


class A2C:
    def __init__(
        self, policy, optimizer, value_loss_coef=0.25, entropy_coef=0.01, max_grad_norm=0.5
    ):
        self.policy = policy
        self.optimizer = optimizer
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm

    def entropy(self, trajectory) -> torch.Tensor:
        logits = trajectory["logits"]
        log_probs = nn.functional.log_softmax(logits, -1)
        probs = log_probs.exp()
        return -(probs * log_probs).sum(dim=1).mean()

    def policy_loss(self, trajectory) -> torch.Tensor:
        # You will need to compute advantages here.
        log_probs = trajectory["log_probs"]
        values = trajectory["values"].detach()
        value_targets = trajectory["value_targets"].detach()
        advantages = value_targets - values
        return -torch.mean(log_probs * advantages)

    def value_loss(self, trajectory) -> torch.Tensor:
        values = trajectory["values"]
        value_targets = trajectory["value_targets"].detach()
        return (values - value_targets).pow(2).mean()

    def grad_norm(self) -> float:
        total_norm = 0.0
        for p in self.policy.model.parameters():
            if p.grad is not None:
                param_norm = p.grad.norm(2)
                total_norm += param_norm.item() ** 2
        grad_norm = total_norm**0.5
        return grad_norm

    def a2c_loss_and_stats(self, trajectory) -> Tuple[torch.Tensor, Dict[str, float]]:
        policy_loss = self.policy_loss(trajectory)
        value_loss = self.value_loss(trajectory)
        entropy = self.entropy(trajectory)
        a2c_loss = policy_loss + self.value_loss_coef * value_loss - entropy * self.entropy_coef
        value_targets = trajectory["value_targets"].detach()
        values = trajectory["values"].detach()
        advantages = value_targets - values
        var_y = value_targets.var(unbiased=False)
        r2 = 1 - (value_targets - values).var(unbiased=False) / (var_y + 1e-8)

        return a2c_loss, {
            "a2c_loss": a2c_loss.item(),
            "policy_loss": policy_loss.item(),
            "value_loss": value_loss.item(),
            "entropy": entropy.item(),
            "r2": r2.item(),
            "value_targets": value_targets.mean().item(),
            "value_predictions": values.mean().item(),
            "advantages": advantages.mean().item(),
        }

    def step(self, trajectory) -> dict[str, float]:
        self.optimizer.zero_grad()
        a2c_loss, stats = self.a2c_loss_and_stats(trajectory)
        a2c_loss.backward()
        grad_norm = self.grad_norm()
        stats["grad_norm"] = grad_norm
        nn.utils.clip_grad_norm_(self.policy.model.parameters(), self.max_grad_norm)
        self.optimizer.step()
        return stats

Now you can train your model. With reasonable hyperparameters training on a single GTX1080 for 10 million steps across all batched environments (which translates to about 5 hours of wall clock time)
it should be possible to achieve *average raw reward over last 100 episodes* (the average is taken over 100 last
episodes in each environment in the batch) of about 600. You should plot this quantity with respect to
`runner.step_var` &mdash; the number of interactions with all environments. It is highly
encouraged to also provide plots of the following quantities (these are useful for debugging as well):

* [Coefficient of Determination](https://en.wikipedia.org/wiki/Coefficient_of_determination) between
value targets and value predictions
* Entropy of the policy $\pi$
* Value loss
* Policy loss
* Value targets
* Value predictions
* Gradient norm
* Advantages
* A2C loss

For optimization we suggest you use RMSProp with learning rate starting from 7e-4 and linearly decayed to 0, smoothing constant (alpha in PyTorch and decay in TensorFlow) equal to 0.99 and epsilon equal to 1e-5.

In [13]:
%load_ext tensorboard
%tensorboard --logdir logs --port 6009 --host 0.0.0.0

Launching TensorBoard...

In [14]:
from tqdm import trange

initial_lr = 7e-4
opt = torch.optim.RMSprop(model.parameters(), lr=initial_lr, alpha=0.99, eps=1e-5)
a2c = A2C(policy, opt)

target_env_steps = 20_000_000
num_updates = target_env_steps // (runner.nsteps * nenvs)

for step in trange(num_updates):
    frac = 1.0 - step / num_updates
    opt.param_groups[0]["lr"] = initial_lr * frac

    trajectory = runner.get_next()
    stats = a2c.step(trajectory)
    for key, value in stats.items():
        env.get_wrapper_attr("add_summary")(key, value)

  1%|          | 1206/100000 [03:22<4:36:36,  5.95it/s]


KeyboardInterrupt: 

### Target networks?

You may recall a technique called "target networks" we used a few weeks ago when we trained a DQN agent to play Atari Breakout and wonder why we have not suggested using them here. The answer is that this is more historical than practical.

While the "chasing the target" problem is still present in actor-critic value estimation and target networks do show up in follow-up papers, the original A3C/A2C papers do not mention them and do not explain this omission.

The hypothesis why this may not be a big deal (compared to Q-learning) goes like this. An A3C/A2C agent selects actions based on policy, not an epsilon greedy exploration function, for which the argmax can change drastically due to tiny errors in function approximation. Therefore, errors in the value target caused by target chasing will cause less damage.

Also, the actor-critic gradient relies on the advantage function $A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$. Compare this to the $Q$-function $Q(s_t, a_t) = r(s_t, a_t) + \gamma \cdot \mathbb{E}_{s_{t+1} \mid s_t, a_t} V(s_{t+1})$ used in Q-learning and SARSA: we would expect that any bias in $V$-function approximation will be carried over from $V(s_{t+1})$ to $V(s_t)$ by gradient updates. However, in the formula for the advantage function the two approximations ($Q$-function and $V$-function) come with opposite signs, and thus the errors will cancel out.

The last reason may be computational. Authors were concerned to beat existent algorithms in the wall-clock learning time, and any overhead of parameter copying (target network update) counted against this goal.